# 🧩 `falcon_tools` — FALC-ON AI's module, written as a notebook

This notebook does not *do* anything. It **defines** things for another notebook to borrow: FALC-ON AI's knowledge, the Class 4 helpers, and the Class 8 classes.

- The Class 9 notebook imports it with `import falcon_tools`.
- Keep this file **in the same folder** as the notebook that imports it — in Colab, upload both to the **Files** panel (📁 in the left sidebar).
- Run it on its own and the self-test at the bottom runs. Import it and that block is skipped.

> 💡 A module is just reusable code you write once and use somewhere else. There is nothing special about this notebook — it is only the *place* the code lives.

In [ ]:
"""FALC-ON AI's toolkit: its knowledge, its helpers, its memory and its assistant."""
import sqlite3

KNOWLEDGE = {
    "class": "The course has 10 classes.",
    "teach": "Alexandra Oliveira and Catarina Santos teach this course.",
    "colab": "We write our code in Google Colab, which is free.",
}

UNKNOWN = "Sorry, I don't know that yet."


def clean_text(text):
    """Return the text in lower case, without spaces at the ends."""
    return text.lower().strip()


def find_answer(question):
    """The stand-in model: answer from KNOWLEDGE, by keyword."""
    for keyword in KNOWLEDGE:
        if keyword in question:
            return KNOWLEDGE[keyword]
    return UNKNOWN

In [ ]:
class ChatMemory:
    """Long-term memory for an assistant, kept in a SQLite table."""

    def __init__(self, filename):
        self._connection = sqlite3.connect(filename)
        self._cursor = self._connection.cursor()
        self._cursor.execute("""
            CREATE TABLE IF NOT EXISTS memory (
                id       INTEGER PRIMARY KEY,
                user     TEXT,
                question TEXT,
                answer   TEXT,
                rating   INTEGER
            )
        """)
        self._cursor.execute("DELETE FROM memory")    # start clean, so re-running is safe
        self._connection.commit()

    def save(self, user, question, answer, rating):
        """Save one conversation turn."""
        self._cursor.execute(
            "INSERT INTO memory (user, question, answer, rating) VALUES (?, ?, ?, ?)",
            (user, question, answer, rating),
        )
        self._connection.commit()

    def low_rated(self):
        """The turns rated below 3 - the answers to improve."""
        self._cursor.execute("SELECT user, question, answer FROM memory WHERE rating < 3")
        return self._cursor.fetchall()

    def close(self):
        """Let go of the database file."""
        self._connection.close()

In [ ]:
class FalconAssistant:
    """FALC-ON AI: answers from its knowledge, and remembers every turn."""

    def __init__(self, name, knowledge, memory):
        self.name = name
        self.knowledge = knowledge
        self.memory = memory

    def answer(self, question):
        """The stand-in model: answer from the knowledge dictionary."""
        for keyword in self.knowledge:
            if keyword in question:
                return self.knowledge[keyword]
        return UNKNOWN

    def chat_turn(self, user, question, rating):
        """Clean the question, answer it, and save the turn in memory."""
        question = clean_text(question)
        reply = self.answer(question)
        self.memory.save(user, question, reply, rating)
        return reply

### The self-test

The block below runs **only when you run this notebook directly**. When another notebook imports `falcon_tools`, it is skipped — so importing stays silent, which is what you want from a module.

`__name__` is a name Python sets for you: it is `"__main__"` in the notebook you actually ran, and the module's own name everywhere else.

In [ ]:
if __name__ == "__main__":
    print("Running falcon_tools directly - self-test:")
    print(clean_text("  What is COLAB?  "))            # what is colab?
    print(find_answer("how many classes?"))            # The course has 10 classes.
    print(find_answer("is there an exam?"))            # Sorry, I don't know that yet.